# A/B analysis

This notebook calls the reusable analysis and power modules. The two-sided test and Newcombe interval are primary; relative lift is secondary. All rows are synthetic, and no historical course data are loaded.

## Primary estimand and inferential plan

The estimand is `p_treatment - p_control`, reported as an absolute percentage-point difference. The prespecified framework is a two-sided pooled two-proportion z-test at alpha 0.05 with a 95% Newcombe hybrid-score interval. The interval receives equal or greater interpretive weight than the p-value.

## Reusable primary analysis

The generated CSV is analyzed through `src.statistics`; this notebook does not duplicate the statistical formulas.

In [ ]:
import json
import pandas as pd
from src.statistics import primary_analysis
from src.power import achieved_power, minimum_detectable_effect, required_sample_size_per_group
from src.reference_statistics import reference_two_proportion_ztest, reference_newcombe_difference_interval
data = pd.read_csv('data/generated/synthetic_ab_experiment.csv')
result = primary_analysis(data)
print(json.dumps(result, indent=2))

The observed difference is +0.86 percentage points and the 95% interval is approximately [-0.31, +2.03] points. Because that interval includes zero, the demonstration does not establish improvement. The +9.0717% relative lift is descriptive context, not the primary estimand or a causal claim.

## Separate in-repository numerical QA

A separately implemented standard-library path recomputes the pooled z statistic, two-sided p-value, and Newcombe interval without calling the primary functions.

In [ ]:
reference_zp = reference_two_proportion_ztest(result['treatment_conversions'], result['treatment_n'], result['control_conversions'], result['control_n'])
reference_ci = reference_newcombe_difference_interval(result['treatment_conversions'], result['treatment_n'], result['control_conversions'], result['control_n'])
print({'reference_z_and_p': reference_zp, 'reference_ci': reference_ci})

## Prospective power and minimum detectable effect

Planning uses a 10% baseline, +1.0-point effect, equal allocation, two-sided alpha 0.05, and 80% desired power. These inputs are declared independently of the observed result; post-hoc observed power is not used as evidence.

In [ ]:
required_n = required_sample_size_per_group(0.10, 0.01, desired_power=0.80)
power_at_5000 = achieved_power(0.10, 0.01, 5000)
mde_at_5000 = minimum_detectable_effect(0.10, 5000, desired_power=0.80)
print({'required_n_per_group': required_n, 'power_at_5000': power_at_5000, 'mde_at_5000': mde_at_5000})

## Decision and limitations

**Statistical decision:** `FAIL_TO_REJECT_H0`. This is not proof of no effect or equality.

**Practical decision:** `UNRESOLVED`. The interval crosses the +1.0-point `PORTFOLIO_SCENARIO_ASSUMPTION`, so it includes effects both below and above the hypothetical threshold.

The analysis is synthetic and fixed-horizon. It does not establish real-user behavior, business ROI, production validity, subgroup effects, or conclusions under repeated unplanned peeking.